## ENEM Microdados → AWS S3

========================

Este script baixa os microdados do ENEM diretamente do site do INEP
e faz upload para o seu bucket S3, organizados por ano.

Estrutura criada no S3:

    s3://<bucket>/raw/enem/ano=2023/microdados_enem_2023.csv
    s3://<bucket>/raw/enem/ano=2024/microdados_enem_2024.csv

Como usar:

    1. Configure o arquivo .env com as credenciais AWS
    2. pip install boto3 requests python-dotenv tqdm
    3. Rode o notebook

In [ ]:
import os
import io
import zipfile
import logging
import requests
import boto3
from dotenv import load_dotenv
from tqdm import tqdm

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
log = logging.getLogger(__name__)

In [ ]:
load_dotenv()

AWS_ACCESS_KEY_ID     = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_REGION            = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
BUCKET_NAME           = os.getenv("S3_BUCKET_NAME")

In [ ]:
ANOS = [2023, 2024]

# URL padrão do INEP (mesmo padrão para todos os anos)
URL_TEMPLATE = "https://download.inep.gov.br/microdados/microdados_enem_{ano}.zip"

In [ ]:
def get_s3_client():
    """Cria e retorna um cliente S3 autenticado."""
    return boto3.client(
        "s3",
        region_name=AWS_REGION,
        aws_access_key_id=AWS_ACCESS_KEY_ID,
        aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    )

In [ ]:
def download_zip_em_memoria(url: str, ano: int) -> bytes:
    """
    Faz download do arquivo ZIP em memória com barra de progresso.
    Retorna os bytes do arquivo ZIP.
    """
    log.info(f"[{ano}] Iniciando download: {url}")

    response = requests.get(url, stream=True, timeout=300)
    response.raise_for_status()

    tamanho_total = int(response.headers.get("content-length", 0))
    tamanho_mb = tamanho_total / (1024 * 1024)
    log.info(f"[{ano}] Tamanho do arquivo: {tamanho_mb:.1f} MB")

    buffer = io.BytesIO()

    with tqdm(
        total=tamanho_total,
        unit="B",
        unit_scale=True,
        desc=f"Baixando ENEM {ano}",
    ) as barra:
        for chunk in response.iter_content(chunk_size=8192):
            buffer.write(chunk)
            barra.update(len(chunk))

    buffer.seek(0)
    log.info(f"[{ano}] Download concluído!")
    return buffer

In [ ]:
def extrair_csv_do_zip(buffer: io.BytesIO, ano: int) -> tuple[str, io.BytesIO]:
    """
    Abre o ZIP em memória e encontra o arquivo CSV principal de microdados.
    Retorna o nome do arquivo CSV e um buffer com seu conteúdo.
    """
    log.info(f"[{ano}] Extraindo CSV do ZIP...")

    with zipfile.ZipFile(buffer) as zf:
        # Lista todos os arquivos dentro do ZIP
        arquivos = zf.namelist()
        log.info(f"[{ano}] Arquivos no ZIP: {arquivos}")

        # Encontra o CSV principal (o maior, que contém os microdados)
        # Os ZIPs do INEP têm um CSV principal com os dados dos participantes
        csvs = [
            f for f in arquivos
            if f.lower().endswith(".csv")
            and "MICRODADOS_ENEM" in f.upper()
        ]

        if not csvs:
            # Fallback: pega qualquer CSV que não seja dicionário ou item
            csvs = [
                f for f in arquivos
                if f.lower().endswith(".csv")
                and "dicionario" not in f.lower()
                and "itens" not in f.lower()
            ]

        if not csvs:
            raise FileNotFoundError(
                f"[{ano}] Nenhum CSV de microdados encontrado no ZIP. "
                f"Arquivos disponíveis: {arquivos}"
            )

        # Pega o maior CSV (que é o arquivo principal de participantes)
        nome_csv = max(csvs, key=lambda f: zf.getinfo(f).file_size)
        log.info(f"[{ano}] CSV encontrado: {nome_csv}")

        csv_buffer = io.BytesIO(zf.read(nome_csv))
        return nome_csv, csv_buffer

In [ ]:
def upload_para_s3(
    s3_client,
    csv_buffer: io.BytesIO,
    nome_csv: str,
    ano: int,
    bucket: str,
):
    """
    Faz upload do CSV para o S3 usando multipart upload (eficiente para arquivos grandes).
    Organiza os dados em partições por ano: raw/enem/ano=XXXX/
    """
    # Chave no S3 com particionamento por ano (padrão Hive, compatível com Glue/Athena)
    s3_key = f"raw/enem/ano={ano}/microdados_enem_{ano}.csv"

    tamanho = csv_buffer.getbuffer().nbytes
    tamanho_mb = tamanho / (1024 * 1024)
    log.info(f"[{ano}] Iniciando upload para s3://{bucket}/{s3_key}")
    log.info(f"[{ano}] Tamanho do CSV: {tamanho_mb:.1f} MB")

    csv_buffer.seek(0)

    # Configuração do multipart upload (recomendado para arquivos > 100MB)
    config = boto3.s3.transfer.TransferConfig(
        multipart_threshold=100 * 1024 * 1024,   # 100 MB
        multipart_chunksize=50 * 1024 * 1024,    # 50 MB por parte
        max_concurrency=4,
    )

    with tqdm(
        total=tamanho,
        unit="B",
        unit_scale=True,
        desc=f"Upload ENEM {ano}",
    ) as barra:
        s3_client.upload_fileobj(
            csv_buffer,
            bucket,
            s3_key,
            Config=config,
            Callback=lambda bytes_transferidos: barra.update(bytes_transferidos),
            ExtraArgs={"ContentType": "text/csv"},
        )

    log.info(f"[{ano}] ✅ Upload concluído: s3://{bucket}/{s3_key}")
    return s3_key

In [ ]:
def verificar_configuracao():
    """Verifica se todas as variáveis de ambiente estão configuradas."""
    erros = []
    if not AWS_ACCESS_KEY_ID:
        erros.append("AWS_ACCESS_KEY_ID não configurado no .env")
    if not AWS_SECRET_ACCESS_KEY:
        erros.append("AWS_SECRET_ACCESS_KEY não configurado no .env")
    if not BUCKET_NAME:
        erros.append("S3_BUCKET_NAME não configurado no .env")
    if erros:
        for erro in erros:
            log.error(erro)
        raise EnvironmentError(
            "Configure o arquivo .env antes de rodar o script. "
            "Veja o .env.example para referência."
        )

In [ ]:
def processar_ano(s3_client, ano: int) -> dict:
    """
    Processa um único ano: download → extração → upload.
    Retorna um dict com o resultado.
    """
    url = URL_TEMPLATE.format(ano=ano)
    resultado = {"ano": ano, "status": "erro", "s3_key": None, "erro": None}

    try:
        # 1. Download do ZIP
        zip_buffer = download_zip_em_memoria(url, ano)

        # 2. Extração do CSV do ZIP
        nome_csv, csv_buffer = extrair_csv_do_zip(zip_buffer, ano)

        # 3. Upload para S3
        s3_key = upload_para_s3(s3_client, csv_buffer, nome_csv, ano, BUCKET_NAME)

        resultado["status"] = "sucesso"
        resultado["s3_key"] = s3_key

    except requests.HTTPError as e:
        resultado["erro"] = f"Erro no download: {e}"
        log.error(f"[{ano}] {resultado['erro']}")

    except FileNotFoundError as e:
        resultado["erro"] = str(e)
        log.error(f"[{ano}] {resultado['erro']}")

    except Exception as e:
        resultado["erro"] = f"Erro inesperado: {e}"
        log.error(f"[{ano}] {resultado['erro']}", exc_info=True)

    return resultado

In [ ]:
def main():
    log.info("=" * 60)
    log.info("  ENEM Microdados → AWS S3")
    log.info("=" * 60)

    # Valida configuração
    verificar_configuracao()

    # Cria cliente S3
    s3_client = get_s3_client()
    log.info(f"Bucket de destino: s3://{BUCKET_NAME}")
    log.info(f"Anos a processar: {ANOS}")
    log.info("")

    # Processa cada ano
    resultados = []
    for ano in ANOS:
        log.info(f"{'─' * 40}")
        log.info(f"Processando ano: {ano}")
        resultado = processar_ano(s3_client, ano)
        resultados.append(resultado)
        log.info("")

    # Resumo final
    log.info("=" * 60)
    log.info("RESUMO FINAL")
    log.info("=" * 60)
    for r in resultados:
        if r["status"] == "sucesso":
            log.info(f"✅ {r['ano']} → s3://{BUCKET_NAME}/{r['s3_key']}")
        else:
            log.error(f"❌ {r['ano']} → FALHOU: {r['erro']}")

    sucessos = sum(1 for r in resultados if r["status"] == "sucesso")
    log.info(f"\n{sucessos}/{len(ANOS)} anos processados com sucesso.")

if __name__ == "__main__":
    main()